In [1]:
!pip install -q datasets pandas matplotlib seaborn wordcloud konlpy scipy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ast import literal_eval
from scipy.stats import chi2_contingency, f_oneway
import re

plt.rcParams['font.family'] = 'NanumGothic'  # 한글 폰트
!apt-get -qq install fonts-nanum > /dev/null
import matplotlib.font_manager as fm
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

from datasets import load_dataset
ds = load_dataset("nvidia/Nemotron-Personas-Korea", split="train")
df = ds.to_pandas()
print(df.shape)
df.head(3)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 13.2 MB/s eta 0:00:00


README.md:   0%|          | 0.00/36.0k [00:00<?, ?B/s]

data/train-00000-of-00009.parquet:   0%|          | 0.00/220M [00:00<?, ?B/s]

data/train-00001-of-00009.parquet:   0%|          | 0.00/220M [00:00<?, ?B/s]

data/train-00002-of-00009.parquet:   0%|          | 0.00/220M [00:00<?, ?B/s]

...

data/train-00007-of-00009.parquet:   0%|          | 0.00/220M [00:00<?, ?B/s]

data/train-00008-of-00009.parquet:   0%|          | 0.00/220M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000000 [00:00<?, ? examples/s]

(1000000, 26)


,uuid,professional_persona,sports_persona,arts_persona,travel_persona,culinary_persona,family_persona,persona,cultural_background,skills_and_expertise,...,marital_status,military_status,family_type,housing_type,education_level,bachelors_field,occupation,district,province,country
0,03b4f36a18e6469386d0286dddd513c8,"전기태 씨는 광주 서구의 하역 현장에서 수십 년간 짐을 쌓아 올리며, 지렛대 원리를...","전기태 씨는 주말마다 무등산 자락을 느릿느릿 걸으며 땀을 흘리고, 내려오는 길에 단...",전기태 씨는 거실 소파에 깊숙이 파묻혀 텔레비전에서 나오는 옛날 가요 프로그램을 보...,전기태 씨는 아내와 함께 전국의 역사 유적지를 찾아다니며 옛 조상들의 발취를 느끼는...,전기태 씨는 일주일에 한 번 배달 짜장면과 탕수육을 시켜 먹는 날을 손꼽아 기다리며...,"전기태 씨는 전·월세 아파트에서 평생의 동반자인 아내와 단출하게 살아가며, 투박한 ...","전기태 씨는 광주 서구에서 평생 하역 일을 하며 살아온 70대 가장으로, 투박한 손...","광주 서구에서 평생을 보내며 투박하지만 정겨운 전라도 사투리가 몸에 배어 있고, 시...",수십 년간 하역 현장에서 다져진 감각으로 짐의 무게 중심을 한눈에 파악해 가장 효율...,...,배우자있음,비현역,배우자와 거주,아파트,초등학교,해당없음,하역 및 적재 관련 단순 종사원,광주-서구,광주,대한민국
1,73f75d42a3934626b0d9a4bff062715a,최은지 씨는 서초동 부동산 사무실에서 장부를 잡으며 복잡한 취득세나 양도세 계산을 ...,최은지 씨는 서초동 예술의전당 근처 산책로를 느릿하게 거닐며 동네 이웃들과 수다를 ...,"최은지 씨는 지역 문화센터의 서예 교실에서 붓을 잡으며 마음을 다스리지만, 쉬는 시...",최은지 씨는 가족들과 함께 경주 불국사의 다보탑이나 부여 낙화암의 절벽을 방문해 옛...,"최은지 씨는 일주일에 대여섯 번은 집 밖에서 식사를 해결하며, 나물 비빔밥이나 청국...",최은지 씨는 남편과 자녀들이 함께 사는 복작복작한 집안 분위기 속에서 가장 영향력 ...,최은지 씨는 서초구에서 부동산 회계 사무원으로 일하며 경제적 자립과 사교적인 삶을 ...,최은지는 서초구의 오래된 다세대 주택가에서 나고 자라며 체면과 질서를 중시하는 분위...,"복잡한 부동산 세금 계산이나 장부 정리를 빠르게 처리하며, 오랜 실무 경험을 바탕으...",...,배우자있음,비현역,배우자·자녀와 거주,다세대주택,4년제 대학교,자연과학·수학,회계 사무원,서울-서초구,서울,대한민국
2,89eca80b88284888ad94c84f56777680,"안상식 씨는 퇴직 후에도 목동 주민센터의 복잡한 서류 뭉치를 보며 흐뭇해하며, 동네...","안상식 씨는 정해진 시간에 집 근처 공원을 천천히 거닐며 나무의 상태를 살피고, 무...","안상식 씨는 나훈아의 구슬픈 가락을 거실에 작게 틀어놓고 멍하니 생각에 잠기거나, ...",안상식 씨는 고등학교 동창들과 함께 창덕궁의 후원을 천천히 걸으며 왕실의 기록을 살...,안상식 씨는 일주일에 한 번 가족들과 함께 동네 단골 한우집에서 육즙 가득한 갈비를...,안상식 씨는 말수 적은 할아버지로서 손주들에게 잔소리 대신 정직하게 살아온 뒷모습을...,"안상식 씨는 목동에서 평생을 성실함과 규칙으로 무장해 온, 꼼꼼한 가계부와 나훈아의...","양천구 목동의 오래된 아파트 단지에서 수십 년째 거주하며, 정해진 시간에 일어나 집...","가계부의 십 원 단위까지 정확하게 맞추고, 집안의 모든 중요 서류를 날짜와 항목별로...",...,배우자있음,비현역,배우자와 거주,아파트,고등학교,해당없음,무직,서울-양천구,서울,대한민국


In [3]:
ojunseo = df[df['uuid'] == '1c48b1e108f04df38ad54716bd7eea07'].iloc[0]

for col in df.columns:
    print(f"[{col}]")
    print(ojunseo[col])
    print()

[uuid]
1c48b1e108f04df38ad54716bd7eea07

[professional_persona]
오준서 씨는 공연의 전체 흐름을 짜는 큐시트 작성과 아티스트 섭외에는 탁월한 감각을 보이지만, 정작 컴퓨터 바탕화면은 이름 없는 폴더와 임시 파일들이 뒤섞여 있어 매번 파일을 찾는 데 한참을 헤맵니다. 현장에서 발생하는 돌발 상황을 유연하게 넘기는 순발력을 갖췄으며, 언젠가 전국의 소극장과 아트홀에 뿌려지는 팸플릿에 자신의 이름이 선명하게 박히는 날을 꿈꾸며 묵묵히 경력을 쌓고 있습니다.

[sports_persona]
오준서 씨는 땀 흘리는 격렬한 운동이나 규칙적인 헬스보다는 계룡산 자락의 한적한 숲길을 느릿하게 산책하며 풍경을 감상하는 시간을 즐깁니다. 정해진 계획에 따라 움직이는 스포츠보다 자신의 속도에 맞춰 걷는 정적인 활동에서 더 큰 편안함을 느끼며 일상의 긴장을 해소합니다.

[arts_persona]
오준서 씨는 주말마다 마음 맞는 친구들과 모여 테라포밍 마스 같은 복잡한 전략 보드게임을 하며 밤을 지새우는 몰입의 시간을 즐깁니다. 평소에는 다양한 음악을 감상하고 영상을 시청하며 기획자로서의 안목을 넓히는 동시에, 사회과학적 시선으로 세상의 소통 방식을 탐구하는 습관이 있습니다.

[travel_persona]
오준서 씨는 화려한 도심의 관광지보다는 친구나 연인과 함께 이름 모를 한적한 숲이나 탁 트인 풍경이 펼쳐진 자연 속으로 떠나는 여행을 선호합니다. 촘촘한 일정표를 짜기보다 발길 닿는 대로 풍경을 감상하며 머릿속을 비워내는 시간을 통해 정서적인 회복을 얻습니다.

[culinary_persona]
오준서 씨는 주 4~6회 정도로 외식이 매우 잦은 편이며, 주로 두툼한 삼겹살이 나오는 한식당이나 육즙 가득한 수제 버거, 페퍼로니 피자 가게를 자주 방문합니다. 배달 음식은 일주일에 한 번 정도로 제한하며, 가급적이면 밖에서 지인들과 어울려 식사하며 잡담을 나누는 시간을 통해 사회적 욕구를 충족합니다.

[family_persona]
